# [A100 40GB] 전체 최종 평가: GRPO v4 step 170 — MMMU test 10,500 / MMMU-Pro 1,730 × 2

**두 계정에 나눠 돌립니다.** 첫 코드 셀의 `PART`만 정하세요.

| `PART` | 계정 | 세트 | 생성 수 | 예상 |
|---|---|---|---|---|
| `'mmmu_test'` | 팀원 계정 | MMMU test 전체 | 10,500 | 약 9~19시간 |
| `'pro'` | 공용 계정 | MMMU-Pro standard(10지선다) 전체 → vision 전체 | 1,730 × 2 | 약 3~6.5시간 |

- 모델: 라운드 GRPO v4 2라운드 step 170 adapter(MMMU val 40GB 596/900)를 이 세션에서 base에 병합(CPU, 약 1~2분). adapter 해시(policy_id)가 val 평가 때와 같은지 확인합니다.
- 문항: `configs/final_eval_ids_full.json`(각 세트 test 분할 전부, 표본 없음). 프롬프트·채점은 450 표본 평가와 같습니다(MMMU test = val과 같은 프롬프트·v2 파서, Pro = 공식 CoT·공식 파서, 무작위 추측 없음).
- 생성 설정은 val 평가와 같습니다(`final_16k.yaml`, seed 3407). Pro는 이미지가 여러 장인 문항이 있어 입력이 길면 `max_model_len`만 늘립니다(기록됨).
- **끊겨도 됩니다:** 256문항 묶음마다 Drive에 저장합니다. 런타임이 끊기면 **같은 PART로 처음부터 다시 실행**하세요. 저장된 문항은 건너뛰고, 문항별 입력 검사(preflight) 결과도 재사용합니다. 재개는 같은 GPU 종류(40GB)에서만 됩니다.
- 진행 로그의 `Saved ... | total n/N | out tok/s | ETA` 줄로 속도와 남은 시간을 볼 수 있습니다. **첫 묶음 몇 개가 끝나면 속도·ETA를 알려 주세요**(유닛 재추정).

> **실행:** 런타임 **A100 GPU** → **런타임 > 모두 실행**. 40GB가 아니면 첫 코드 셀에서 멈춥니다(연결 해제 및 삭제 후 다시 연결). **4번 셀이 도는 동안 다른 셀을 실행하지 마세요.** 마지막 셀이 런타임을 자동 종료합니다.
>
> **실행 전 Drive 업로드(공용 계정 Drive, 팀원 계정은 공유 폴더 바로가기로 같은 경로):** `MyDrive/MMDL/code/qwen3_vl_mmmu_eval/` 아래에 ① `src/mmmu_final/` 폴더(파이썬 파일 6개, 2026-11-21 수정본) ② `configs/final_eval_ids_full.json`. 첫 코드 셀이 로컬 검증본과 해시를 비교합니다. step 170 adapter는 `MyDrive/VIRL39K_RFT_4B/round_grpo_v4/round_02/train/checkpoint_00680_00170/`에 있어야 합니다.

In [3]:
from google.colab import drive
from pathlib import Path
import os, sys, json, re, subprocess, shutil, hashlib, time

drive.mount('/content/drive')

# === 사용자 설정: 이 값만 바꿉니다 ===
PART = 'pro'   # 'mmmu_test' (팀원 계정: MMMU test 10,500) 또는 'pro' (공용 계정: MMMU-Pro standard + vision 1,730씩)
CHUNK_SIZE = 256   # 묶음 크기(저장 단위). 바꾸지 않아도 됩니다.

assert PART in ('mmmu_test', 'pro'), "첫 줄의 PART를 'mmmu_test'(팀원 계정) 또는 'pro'(공용 계정)로 정하세요."
SETS = {'mmmu_test': ['mmmu_test'], 'pro': ['pro_standard', 'pro_vision']}[PART]
MODEL_TAG = 'grpo_v4_step170'
N_EXPECTED = {'mmmu_test': 10500, 'pro_standard': 1730, 'pro_vision': 1730}

PROJECT = Path('/content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval')
CONFIG = PROJECT/'configs'/'final_16k.yaml'
IDS = PROJECT/'configs'/'final_eval_ids_full.json'
REQ = PROJECT/'requirements.txt'
BASELINE_RUN_DIR = PROJECT/'outputs'/'final_seed3407_16k'                            # chat template 비교용 (val baseline)
MMMU_CACHE = Path('/content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_dataset_cache')   # MMMU test는 기존 Drive 캐시
PRO_CACHE = Path('/content/mmmu_pro_cache')                                     # MMMU-Pro는 Colab 로컬
OUTPUT_ROOT = Path('/content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs/final_eval_full_a100_40gb')
EVAL_VENV = Path('/content/mmmu_eval_venv')
REQUIRED_GPU = 'NVIDIA A100-SXM4-40GB'
BASE_MODEL_ID, BASE_REVISION = 'Qwen/Qwen3-VL-4B-Instruct', 'ebb281ec70b05090aa6165b016eac8ec08e71b17'

# --- GRPO v4 step 170 adapter (MMMU val 40GB 596/900에서 평가한 것과 같은 파일인지) ---
GRPO_RUN_DIR = Path('/content/drive/MyDrive/VIRL39K_RFT_4B/round_grpo_v4')
PV_FILE = GRPO_RUN_DIR/'validation'/'round_02_step170'/'posthoc_verdict.json'
EXPECTED_POLICY_ID = '55922b8dfc2948d12f6e5190f02be0dd92804aa2ba0b78a19d2a3d533bcfc442'
assert PV_FILE.is_file(), f'step 170 사후 검증 파일이 없습니다: {PV_FILE} (팀원 계정이면 VIRL39K_RFT_4B 공유·바로가기 확인)'
pv = json.loads(PV_FILE.read_text())
assert pv['candidate'] == 'round_02_step170' and pv['passed'] and pv['policy_id'] == EXPECTED_POLICY_ID, f'예상한 step 170 후보가 아닙니다: {pv.get("policy_id")}'
ADAPTER_DIR = Path(pv['adapter'])
def _sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(2**20), b''): h.update(b)
    return h.hexdigest()
POLICY_ID = hashlib.sha256(json.dumps({n: _sha(ADAPTER_DIR/n) for n in ('adapter_config.json', 'adapter_model.safetensors')},
                                      sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
assert POLICY_ID == EXPECTED_POLICY_ID, 'adapter 파일이 MMMU val 596 평가 때와 다릅니다(해시 불일치).'
print('✔ GRPO v4 step 170 adapter', POLICY_ID[:16], '|', ADAPTER_DIR)
LOCAL_MODEL = Path(f'/content/grpo_v4_step170_{POLICY_ID[:12]}_merged_for_final_eval')

# --- 고정 파일 확인: 평가 설정·문항 ID·mmmu_final 코드가 로컬 드라이런 통과본과 같은지 ---
EXPECTED_SHA = {'final_16k.yaml': 'a6b0ac30c57e5d2a4358233060cd45ff642a3ba2193eeeecbeea45486334d818',
                'final_eval_ids_full.json': 'b0fdacad4993c1556eee93395a202be86fcf9ea4e3a6f25011288a53360d4c66'}
for p in (CONFIG, IDS, REQ, PROJECT/'src'/'mmmu_final'/'run.py'):
    assert p.is_file(), f'필요한 파일이 없습니다: {p}  → Drive의 평가 프로젝트에 로컬 최신본을 올리세요'
for p in (CONFIG, IDS):
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    assert h == EXPECTED_SHA[p.name], f'{p.name}이 고정본과 다릅니다: {h}'
EXPECTED_FINAL_SRC = {'__init__.py': '39a51674e8b071b4def2f6385ac6d4fa6db778d4b8266558aae4f82688e1a382', 'compare.py': 'c7dd273e39a202969d946cb788dae4500714fd9cea1a112ef509f8ef86166b19', 'official_pro.py': 'd3f47f4440f46d132f77b2f708593fccd9aa9ded6892f9e0677e3a9c0df00cfb', 'run.py': '90c6aa73eccfd806f91028d2447be139b667aac8467c58fecae983675746a3bd', 'scoring.py': '8765a250185fa97234804abb58d2900a2261e0934d168a20c9cd59c1a75531f1', 'sets.py': '064d1061b23ee1d1005d3d1b8094b0797b8ee697d2efcbc85be74566d5b08e0d'}
got = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (PROJECT/'src'/'mmmu_final').glob('*.py')}
bad = sorted(n for n in EXPECTED_FINAL_SRC if got.get(n) != EXPECTED_FINAL_SRC[n])
assert not bad, f'src/mmmu_final/의 다음 파일이 검증본과 다릅니다(또는 없음): {bad} → 2026-11-21 수정본을 다시 올리세요'
print('✔ 평가 설정·전체 문항 ID·mmmu_final 코드 고정본 확인')

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
assert gpu_name == REQUIRED_GPU, (f'최종 평가는 {REQUIRED_GPU}에서만 돌립니다. 현재: {gpu_name}\n'
    '→ 런타임 > 런타임 연결 해제 및 삭제 후 다시 연결해 40GB를 받으세요.')

# --- 결과 폴더 쓰기 확인 (팀원 계정은 공유 폴더에 편집 권한이 있어야 함) ---
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
probe = OUTPUT_ROOT/f'.write_test_{PART}'; probe.write_text('ok'); probe.unlink()
# --- 이전 시도가 시작 직후 끊겨 잠금 파일만 남은 세트 폴더 정리 (평가 프로세스가 없을 때만) ---
for s in SETS:
    d = OUTPUT_ROOT/MODEL_TAG/s
    if d.is_dir() and {p.name for p in d.iterdir()} <= {'.run.lock'}:
        running = subprocess.run(['pgrep', '-f', 'mmmu_final[.]run'], capture_output=True).returncode == 0
        assert not running, '평가 프로세스가 아직 돌고 있습니다. !pkill -f mmmu_final.run 으로 종료한 뒤 다시 실행하세요.'
        shutil.rmtree(d); print('잠금 파일만 남은 폴더를 지웠습니다:', d)
    elif (d/'run_meta.json').is_file():
        m = json.loads((d/'run_meta.json').read_text())
        n = sum(1 for _ in open(d/'raw_outputs.jsonl', encoding='utf-8')) if (d/'raw_outputs.jsonl').is_file() else 0
        print(f'[{s}] 이전 실행 기록 있음: 저장된 응답 약 {n}/{N_EXPECTED[s]}, 상태 {m.get("status")} → 이어서 진행')
print('GPU:', gpu_name, '| PART:', PART, SETS, '| 결과:', OUTPUT_ROOT/MODEL_TAG)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✔ GRPO v4 step 170 adapter 55922b8dfc2948d1 | /content/drive/MyDrive/VIRL39K_RFT_4B/round_grpo_v4/round_02/train/checkpoint_00680_00170
✔ 평가 설정·전체 문항 ID·mmmu_final 코드 고정본 확인
GPU: NVIDIA A100-SXM4-40GB | PART: pro ['pro_standard', 'pro_vision'] | 결과: /content/drive/MyDrive/MMDL/pmc_vqa/eval/mmmu_eval_outputs/final_eval_full_a100_40gb/grpo_v4_step170


## 1. 평가 전용 가상환경 (기존 평가 노트북과 같은 셀)

In [4]:
def checked(command, *, cwd=None, env=None):
    print('$', ' '.join(map(str, command)), flush=True)
    return subprocess.run([str(v) for v in command], cwd=cwd, env=env, check=True)

EVAL_PY = EVAL_VENV/'bin'/'python'
marker = EVAL_VENV/'.installed_ok'
req_hash = hashlib.sha256(REQ.read_bytes()).hexdigest()
if not (marker.is_file() and marker.read_text() == req_hash):
    shutil.rmtree(EVAL_VENV, ignore_errors=True)
    checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    checked([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(EVAL_VENV)])
    print('requirements:\n' + REQ.read_text(encoding='utf-8'))
    checked([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(EVAL_PY), '-r', str(REQ)])
    marker.write_text(req_hash)
out = subprocess.run([str(EVAL_PY), '-c',
    'import sys,torch,vllm,transformers,datasets,qwen_vl_utils,importlib.metadata as m;'
    'print(sys.version.split()[0], torch.__version__, vllm.__version__, transformers.__version__, '
    'datasets.__version__, m.version("qwen-vl-utils"), torch.cuda.is_available())'],
    capture_output=True, text=True, check=True).stdout.strip()
print('평가 환경 (python torch vllm transformers datasets qwen-vl-utils cuda):', out)
pyv, torchv, vllmv, tfv, dsv, qvlv, cuda = out.split()
assert (torchv.split('+')[0], vllmv, tfv, dsv, qvlv) == ('2.8.0', '0.11.0', '4.57.1', '4.2.0', '0.0.14'), 'baseline 실행 환경과 패키지 버전이 다릅니다'
assert cuda == 'True', '평가 환경에서 CUDA를 찾지 못했습니다'
print('✔ baseline과 같은 패키지 버전')

$ /usr/bin/python3 -m pip install -q uv
$ /usr/bin/python3 -m uv venv --python 3.12 /content/mmmu_eval_venv
requirements:
# Candidate pins; NOT yet GPU-validated. See README and Colab verification.
# After successful A100 smoke/resume tests, capture the installed environment
# using scripts/freeze_verified_environment.py and review/commit the lock.
vllm==0.11.0
torch==2.8.0
torchvision==0.23.0
torchaudio==2.8.0
transformers==4.57.1
qwen-vl-utils==0.0.14
datasets==4.2.0
Pillow==11.3.0
PyYAML==6.0.3
pytest==8.4.2

$ /usr/bin/python3 -m uv pip install --python /content/mmmu_eval_venv/bin/python -r /content/drive/MyDrive/MMDL/code/qwen3_vl_mmmu_eval/requirements.txt
평가 환경 (python torch vllm transformers datasets qwen-vl-utils cuda): 3.12.3 2.8.0+cu128 0.11.0 4.57.1 4.2.0 0.0.14 True
✔ baseline과 같은 패키지 버전


## 2. step 170 adapter 병합 (CPU, Colab 로컬, GRPO val 평가와 같은 스크립트)

In [6]:
def check_weights(model_dir, fix=False):
    files = {p.name for p in model_dir.glob('*.safetensors')}
    assert files, f'*.safetensors 가중치 없음: {model_dir}'
    index = model_dir/'model.safetensors.index.json'
    if not index.is_file():
        assert files == {'model.safetensors'}, f'index 없이 여러 shard: {files}'
        return
    need = set(json.loads(index.read_text())['weight_map'].values())
    missing = need - files
    if missing and fix:
        for name in sorted(missing):
            stem = name[:-len('.safetensors')]
            alt = [f for f in files if re.fullmatch(re.escape(stem) + r'[-_ ]?\(?\d{1,3}\)?\.safetensors', f)]
            assert len(alt) == 1, f'{name}에 해당하는 파일을 하나로 특정할 수 없음: {alt}'
            print(f'  파일명 복구: {alt[0]} -> {name}')
            (model_dir/alt[0]).rename(model_dir/name)
        files = {p.name for p in model_dir.glob('*.safetensors')}
        missing = need - files
    assert not missing, f'index가 요구하는 가중치 파일 없음: {missing} (있는 파일: {sorted(files)})'
    extra = files - need
    assert not extra, f'index에 없는 가중치 파일이 섞여 있음: {extra}'

MERGE_PY = Path('/content/grpo_merge.py')
MERGE_PY.write_text('''import json, os, sys, torch
from pathlib import Path
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import PeftModel
adapter, out, model_id, rev, info = sys.argv[1], Path(sys.argv[2]), sys.argv[3], sys.argv[4], json.loads(sys.argv[5])
tmp = out.with_name(out.name + '.building')
base = Qwen3VLForConditionalGeneration.from_pretrained(model_id, revision=rev, dtype=torch.bfloat16, device_map={'': 'cpu'}, low_cpu_mem_usage=True)
merged = PeftModel.from_pretrained(base, adapter, is_trainable=False).merge_and_unload(safe_merge=True)
merged.config.use_cache = True
merged.save_pretrained(tmp, safe_serialization=True, max_shard_size='4GB')
AutoProcessor.from_pretrained(model_id, revision=rev).save_pretrained(tmp)
(tmp / 'merge_info.json').write_text(json.dumps(info, indent=2))
os.replace(tmp, out)
print('merged:', out)
''')

merge_info = {'base': BASE_MODEL_ID, 'base_revision': BASE_REVISION, 'adapter': str(ADAPTER_DIR), 'policy_id': POLICY_ID,
              'method': 'virl39k_round_grpo', 'run_tag': 'round_grpo_v4', 'selected_name': 'round_02_step170',
              'posthoc_verdict_sha256': _sha(PV_FILE)}
if (LOCAL_MODEL/'merge_info.json').is_file():
    old = json.loads((LOCAL_MODEL/'merge_info.json').read_text())
    assert old['policy_id'] == POLICY_ID, f'로컬 병합 모델이 다른 adapter입니다: {old}'
    print('이미 병합된 로컬 모델 재사용:', LOCAL_MODEL)
else:
    shutil.rmtree(LOCAL_MODEL.with_name(LOCAL_MODEL.name + '.building'), ignore_errors=True)
    assert shutil.disk_usage('/content').free > 20 * 2**30, '로컬 디스크 여유 공간 부족(병합에 약 9GB 필요)'
    checked([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(EVAL_PY), '--no-deps', 'peft==0.17.1', 'accelerate==1.11.0'])
    vers = subprocess.run([str(EVAL_PY), '-c', 'import torch,vllm,transformers,peft;print(torch.__version__,vllm.__version__,transformers.__version__,peft.__version__)'],
                          capture_output=True, text=True, check=True).stdout.split()
    assert (vers[0].split('+')[0], vers[1], vers[2], vers[3]) == ('2.8.0', '0.11.0', '4.57.1', '0.17.1'), f'평가 환경 버전이 바뀌었습니다: {vers}'
    t0 = time.time()
    checked([EVAL_PY, MERGE_PY, ADAPTER_DIR, LOCAL_MODEL, BASE_MODEL_ID, BASE_REVISION, json.dumps(merge_info)])
    print(f'병합 {time.time()-t0:.0f}s')
check_weights(LOCAL_MODEL, fix=True)
tmpl = subprocess.run([str(EVAL_PY), '-c',
    'import sys;from transformers import AutoProcessor;'
    'sys.stdout.write(AutoProcessor.from_pretrained(sys.argv[1]).chat_template)', str(LOCAL_MODEL)],
    capture_output=True, text=True, check=True).stdout
assert tmpl == (BASELINE_RUN_DIR/'chat_template.jinja').read_text(encoding='utf-8'), '병합 모델의 chat template이 val baseline과 다릅니다.'
print('✔ 모델:', LOCAL_MODEL, '| chat template = val baseline (sha256', hashlib.sha256(tmpl.encode()).hexdigest()[:16] + ')')

이미 병합된 로컬 모델 재사용: /content/grpo_v4_step170_55922b8dfc29_merged_for_final_eval
✔ 모델: /content/grpo_v4_step170_55922b8dfc29_merged_for_final_eval | chat template = val baseline (sha256 3636d0f0bd6bef02)


## 3. 세트별 생성 + 채점
`PART`의 세트를 차례로 `python -m mmmu_final.run`으로 실행합니다(묶음마다 저장, 끊기면 다시 실행해서 이어 감). 전체 로그는 `OUTPUT_ROOT/grpo_v4_step170/<세트>/stdout.log`.

In [7]:
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT/'src'); env['PYTHONUNBUFFERED'] = '1'; env['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
progress = re.compile(r'\d+%\|')
for s in SETS:
    out_dir = OUTPUT_ROOT/MODEL_TAG/s
    if (out_dir/'scores.json').is_file():
        sc = json.loads((out_dir/'scores.json').read_text())
        print(f'[{s}] 이미 채점 완료 → 건너뜀: {sc["correct"]}/{sc["n"]}'); continue
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(EVAL_PY), '-m', 'mmmu_final.run', '--set', s, '--model_tag', MODEL_TAG, '--model_path', str(LOCAL_MODEL),
           '--config', str(CONFIG), '--ids', str(IDS), '--output_root', str(OUTPUT_ROOT),
           '--data_root', str(MMMU_CACHE if s == 'mmmu_test' else PRO_CACHE), '--seed', '3407',
           '--require_gpu', 'A100-SXM4-40GB', '--chunk_size', str(CHUNK_SIZE)]
    print(f'\n=== [{s}] 시작 ({N_EXPECTED[s]}문항) ===', flush=True); t0 = time.time(); last = 0.0
    with open(out_dir/'stdout.log', 'a', encoding='utf-8') as log:
        proc = subprocess.Popen(cmd, cwd=PROJECT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line); log.flush()
            if progress.search(line) or line.startswith('Preflight') and 'questions' not in line:
                if time.time() - last < 120: continue
                last = time.time()
            print(line, end='', flush=True)
        rc = proc.wait()
    print(f'[{s}] 이번 세션 {(time.time()-t0)/3600:.2f}시간, 종료 코드 {rc}')
    if rc != 0:
        raise RuntimeError(f'[{s}] 평가 실패 (exit {rc}). 로그: {out_dir/"stdout.log"} — 끊긴 것이면 같은 PART로 처음부터 다시 실행하면 이어 갑니다.')
print('\n✔ 완료:', SETS)


=== [pro_standard] 시작 (1730문항) ===
GPU: NVIDIA A100-SXM4-40GB

Generating test split:   0%|          | 0/1730 [00:00<?, ? examples/s]
/content/mmmu_eval_venv/lib/python3.12/site-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Preflight pro_standard: Physics
Preflight: 1730 questions, longest input 9487 tokens -> max_model_len 26624
INFO 11-21 18:05:53 [__init__.py:216] Automatically detected platform cuda.
INFO 11-21 18:05:56 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'seed': 3407, 'max_model_len': 26624, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 10}, 'generation_config': 'vllm', 'model': '/content/grpo_v4_step170_55922b8dfc29_merged_for_final_eval'}
INFO 11-21 18:06:09 [model.py:547] Resolved architecture: Qwen3VLForConditionalGeneration
`torch_dtype` is deprecated! Use `dtype` instead!
INFO 11-21 18:06:09 [model.py:1510] Using max model len 26624
INFO 11-21 

## 4. 결과 요약

In [8]:
for s in N_EXPECTED:
    f = OUTPUT_ROOT/MODEL_TAG/s/'scores.json'
    if not f.is_file():
        print(f'{s:<13} (결과 없음 — 다른 계정에서 진행 중이거나 미실행)'); continue
    sc = json.loads(f.read_text()); meta = json.loads((f.parent/'run_meta.json').read_text())
    assert sc['n'] == N_EXPECTED[s], sc['n']
    extra = (f" | 공식(무작위 포함, 참고) {sc['reference_official_with_guess']['correct']} | 출처별 {sc['by_origin']}" if 'by_origin' in sc else
             f" | 객관식/주관식 {sc['question_type']}")
    print(f"{s:<13} {sc['correct']}/{sc['n']} = {100*sc['accuracy']:.2f}% | 잘림 {sc['trunc_pct']:.1f}% | 중앙 {sc['median_output_tokens']:.0f}토큰 | "
          f"max_model_len {sc['max_model_len_used']} | {sc['gpu']} | 생성 {meta.get('generation_seconds', 0)/3600:.2f}시간{extra}")

mmmu_test     (결과 없음 — 다른 계정에서 진행 중이거나 미실행)
pro_standard  937/1730 = 54.16% | 잘림 3.3% | 중앙 716토큰 | max_model_len 26624 | NVIDIA A100-SXM4-40GB | 생성 2.29시간 | 공식(무작위 포함, 참고) 945 | 출처별 {'test': {'n': 1153, 'correct': 634}, 'validation': {'n': 577, 'correct': 303}}
pro_vision    881/1730 = 50.92% | 잘림 3.8% | 중앙 890토큰 | max_model_len 24576 | NVIDIA A100-SXM4-40GB | 생성 2.95시간 | 공식(무작위 포함, 참고) 894 | 출처별 {'test': {'n': 1153, 'correct': 596}, 'validation': {'n': 577, 'correct': 285}}


## 5. 결과 보존
Drive의 `mmmu_eval_outputs/final_eval_full_a100_40gb/` 폴더 전체를 로컬 `MMDL/pmc_vqa/eval/mmmu_eval_outputs/`로 가져와 주세요. 응답 원문(`raw_outputs.jsonl`)에는 MMMU·MMMU-Pro 문항 내용이 들어 있으니 공개 저장소에는 집계 파일(`scores.json`)만 올립니다.

In [9]:
import time
from google.colab import runtime
print('모든 셀 완료. 60초 뒤 런타임을 종료합니다. 중단하려면 셀을 정지하세요.')
time.sleep(60)
runtime.unassign()

모든 셀 완료. 60초 뒤 런타임을 종료합니다. 중단하려면 셀을 정지하세요.
